## Imports

In [6]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import EXP_neuro_fuzzy_toolbox as nft

In [7]:
import numpy as np

In [8]:
from sklearn.preprocessing import MinMaxScaler

# Surface (1k)

## Data

In [9]:
def z(x, y):
  return ((3) * ((1-x)**2) * (np.exp(-(x**2)-((y+1)**2))) - (10) * ((x/5)-(x**3)-(y**5)) * (np.exp(-(x**2)-(y**2))) - (1/3)*np.exp(-(x+1)**2-(y**2)))

#Training
x0 = np.random.uniform(-3,3,1000)
x1 = np.random.uniform(-3,3,1000)

e = np.random.normal(0,0.5,1000) #noise
Y = z(x0,x1) + e

#Testing
x0_test = np.random.uniform(-3,3,1000)
x1_test = np.random.uniform(-3,3,1000)

Y_test = z(x0_test,x1_test)

In [10]:
#Training
scaler = MinMaxScaler(feature_range=(0, 1))
vstack_train = np.vstack((x0,x1)).T
scaled_train = scaler.fit_transform(vstack_train)

#Testing
vstack_test = np.vstack((x0_test,x1_test)).T
scaled_test = scaler.transform(vstack_test)

In [11]:
dtype = torch.float64

train_loader = data.DataLoader(
    data.TensorDataset(
        torch.tensor(scaled_train, dtype=dtype), 
        torch.tensor(Y, dtype=dtype)), 
    batch_size = 8,
    shuffle = True)

x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

x_test = torch.tensor(scaled_test, dtype=dtype)
y_test = torch.tensor(Y_test, dtype=dtype)

In [12]:
x_train.dtype

torch.float64

## Model & Training

### ANFIS

In [13]:
model = nft.rule_reduced_ANFIS(
    input_size = 2,
    num_mfs = 2,
    outputs = 1,
    dtype=torch.float64
)

In [14]:
model.init_premises(x_train)

In [15]:
model.init_consequents(x_train, y_train)

### Hybrid Learning Algorithm

In [16]:
loss_fn = nn.functional.mse_loss
#loss_fn = nn.functional.binary_cross_entropy
#loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.0001, 'weight_decay': 0.001}

early_stopping = nft.EarlyStopping(patience=30, delta=0.001)

In [17]:
trainer = nft.Basic_optimizer_training_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [18]:
Ngrow = 60
dGrow = 0.7
Nsplit = 50
eSplit = 0.8
Nvanish = 6
lVanish = 3

max_iterations = 40

anfis_trainer = trainer

validation = 0.3
sonfis_early_stopping = nft.EarlyStopping(patience=8)
last_training_iteration = True

In [19]:
sonfis = nft.SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [20]:
%time sonfis(model, train_loader, verbose=True)

Iteration:  0/40 - loss: 2.231676 - validation loss: 2.492280
 -> ANFIS rules: 2

Iteration:  1/40 - loss: 1.776583 - validation loss: 2.150732
 -> ANFIS rules: 4

Iteration:  2/40 - loss: 1.700710 - validation loss: 2.061545
 -> ANFIS rules: 6

Iteration:  3/40 - loss: 1.618899 - validation loss: 1.966571
 -> ANFIS rules: 8

Iteration:  4/40 - loss: 1.605048 - validation loss: 1.948965
 -> ANFIS rules: 8

Iteration:  5/40 - loss: 1.604232 - validation loss: 1.946997
 -> ANFIS rules: 9

Iteration:  6/40 - loss: 1.536202 - validation loss: 1.865069
 -> ANFIS rules: 10

Iteration:  7/40 - loss: 1.506082 - validation loss: 1.850462
 -> ANFIS rules: 11

Iteration:  8/40 - loss: 0.961141 - validation loss: 1.144280
 -> ANFIS rules: 13

Iteration:  9/40 - loss: 0.803961 - validation loss: 0.921095
 -> ANFIS rules: 14

Iteration: 10/40 - loss: 0.803961 - validation loss: 0.921095
 -> ANFIS rules: 15

Iteration: 11/40 - loss: 0.817620 - validation loss: 0.937013
 -> ANFIS rules: 15

Iteration:

In [21]:
test_measures = nft.get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

MSE: 0.23819649902675824
RMSE: 0.48805378702224844
MAE: 0.3374047340167836
R2: 0.9265173533946554
MAPE: 31.983718848804994


In [22]:
train_measures = nft.get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

MSE: 0.38634429784119456
RMSE: 0.6215660044123991
MAE: 0.4759731824800369
R2: 0.9003706732780778
MAPE: 5.625070554653894
